# Isolated Samples Analysis for Optics Express Paper

This notebook performs comprehensive analysis on isolated samples including:
1. Overall Segmentation Performance
2. Per-Class Performance Metrics
3. Per-Sample Detailed Results
4. Tissue Area Comparison
5. Boundary Metrics (Hausdorff Distance, Average Surface Distance)
6. Failure Case Analysis

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import directed_hausdorff
from scipy.ndimage import distance_transform_edt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set publication-quality defaults
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
class Config:
    """Configuration for analysis"""
    
    # Paths
    MODEL_PATH = Path("models/best_model.pth")
    DATA_SPLIT_PATH = Path("models/data_split.json")
    OUTPUT_DIR = Path("paper_analysis_results")
    
    # Model settings
    ENCODER_NAME = 'resnet34'
    INPUT_SIZE = (512, 512)
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Class names
    CLASS_NAMES = ['Background', 'Tissue', 'OS', 'Vaginal']
    CLASS_COLORS = {
        0: [0, 0, 0],        # Background - black
        1: [0, 0, 255],      # Tissue - blue
        2: [0, 255, 0],      # OS - green
        3: [255, 0, 0]       # Vaginal - red
    }
    
    # Analysis settings
    HAUSDORFF_PERCENTILE = 95
    CONFIDENCE_LEVEL = 0.95

# Create output directory
Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Configuration set")
print(f"  Device: {Config.DEVICE}")
print(f"  Output directory: {Config.OUTPUT_DIR}")

## 3. Model Architecture (Same as Training)

In [ ]:
class UNetWithPretrainedEncoder(nn.Module):
    """U-Net with pretrained encoder - same as training"""
    
    def __init__(self, encoder_name='resnet34', num_classes=4):
        super().__init__()
        
        self.encoder_name = encoder_name
        self.num_classes = num_classes
        
        if encoder_name == 'resnet34':
            encoder = models.resnet34(weights=None)
            encoder_channels = [64, 64, 128, 256, 512]
        elif encoder_name == 'resnet50':
            encoder = models.resnet50(weights=None)
            encoder_channels = [64, 256, 512, 1024, 2048]
        else:
            raise ValueError(f"Unsupported encoder: {encoder_name}")
        
        self.encoder0 = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.encoder1 = nn.Sequential(encoder.maxpool, encoder.layer1)
        self.encoder2 = encoder.layer2
        self.encoder3 = encoder.layer3
        self.encoder4 = encoder.layer4
        
        self.enc_channels = encoder_channels
        
        self.decoder4 = self._decoder_block(encoder_channels[4] + encoder_channels[3], encoder_channels[3])
        self.decoder3 = self._decoder_block(encoder_channels[3] + encoder_channels[2], encoder_channels[2])
        self.decoder2 = self._decoder_block(encoder_channels[2] + encoder_channels[1], encoder_channels[1])
        self.decoder1 = self._decoder_block(encoder_channels[1] + encoder_channels[0], encoder_channels[0])
        self.decoder0 = self._decoder_block(encoder_channels[0], 64)
        
        self.final_conv = nn.Conv2d(64, num_classes, 1)
    
    def _decoder_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        enc0 = self.encoder0(x)
        enc1 = self.encoder1(enc0)
        enc2 = self.encoder2(enc1)
        enc3 = self.encoder3(enc2)
        enc4 = self.encoder4(enc3)
        
        dec4 = F.interpolate(enc4, size=enc3.shape[2:], mode='bilinear', align_corners=True)
        dec4 = torch.cat([dec4, enc3], dim=1)
        dec4 = self.decoder4(dec4)
        
        dec3 = F.interpolate(dec4, size=enc2.shape[2:], mode='bilinear', align_corners=True)
        dec3 = torch.cat([dec3, enc2], dim=1)
        dec3 = self.decoder3(dec3)
        
        dec2 = F.interpolate(dec3, size=enc1.shape[2:], mode='bilinear', align_corners=True)
        dec2 = torch.cat([dec2, enc1], dim=1)
        dec2 = self.decoder2(dec2)
        
        dec1 = F.interpolate(dec2, size=enc0.shape[2:], mode='bilinear', align_corners=True)
        dec1 = torch.cat([dec1, enc0], dim=1)
        dec1 = self.decoder1(dec1)
        
        dec0 = F.interpolate(dec1, size=x.shape[2:], mode='bilinear', align_corners=True)
        dec0 = self.decoder0(dec0)
        
        out = self.final_conv(dec0)
        
        return out

print("✓ Model architecture defined")

## 4. Data Loading Functions

In [ ]:
def extract_m11_and_mask(npz_path: Path) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    """Extract M11 and 4-class mask from NPZ file."""
    try:
        with np.load(npz_path, allow_pickle=True) as data:
            # Extract M11
            m11 = None
            if 'nM11s' in data:
                m11 = np.array(data['nM11s'])
            elif 'M11s' in data:
                m11_raw = np.array(data['M11s'])
                m11_min, m11_max = m11_raw.min(), m11_raw.max()
                if m11_max > m11_min:
                    m11 = (m11_raw - m11_min) / (m11_max - m11_min)
                else:
                    m11 = np.zeros_like(m11_raw, dtype=np.float32)
            elif 'nM' in data:
                nM = np.array(data['nM'])
                if nM.ndim == 4 and nM.shape[-2:] == (4, 4):
                    m11 = nM[:, :, 0, 0]
                elif nM.ndim == 3 and nM.shape[-1] == 16:
                    m11 = nM[:, :, 0]
            
            if m11 is None or m11.ndim != 2:
                return None
            
            # Extract masks
            tissue_mask = None
            for key in ['tissue_mask', 'annotation_mask']:
                if key in data:
                    mask_data = data[key]
                    if isinstance(mask_data, np.ndarray) and mask_data.size > 0:
                        tissue_mask = np.array(mask_data) > 0
                        break
            
            if tissue_mask is None:
                return None
            
            os_mask = None
            if 'os_mask' in data:
                mask_data = data['os_mask']
                if isinstance(mask_data, np.ndarray) and mask_data.size > 0:
                    os_mask = np.array(mask_data) > 0
            
            vaginal_mask = None
            if 'vaginal_mask' in data:
                mask_data = data['vaginal_mask']
                if isinstance(mask_data, np.ndarray) and mask_data.size > 0:
                    vaginal_mask = np.array(mask_data) > 0
            
            # Combine into 4-class mask
            combined_mask = np.zeros_like(tissue_mask, dtype=np.int64)
            combined_mask[tissue_mask] = 1
            if os_mask is not None:
                combined_mask[os_mask] = 2
            if vaginal_mask is not None:
                combined_mask[vaginal_mask] = 3
            
            return m11.astype(np.float32), combined_mask.astype(np.int64)
    
    except Exception as e:
        print(f"Error loading {npz_path}: {e}")
        return None


def preprocess_m11(m11: np.ndarray, target_size: Tuple[int, int]) -> torch.Tensor:
    """Preprocess M11 for model input."""
    # Resize if needed
    if m11.shape != target_size:
        m11_torch = torch.from_numpy(m11).float().unsqueeze(0).unsqueeze(0)
        m11_torch = F.interpolate(m11_torch, size=target_size, mode='bilinear', align_corners=True)
        m11 = m11_torch.squeeze().numpy()
    
    # Convert to 3-channel RGB
    m11_rgb = np.stack([m11, m11, m11], axis=0)
    
    # Normalize with ImageNet stats
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    m11_rgb = (m11_rgb - mean) / std
    
    tensor = torch.from_numpy(m11_rgb).float().unsqueeze(0)
    
    return tensor

print("✓ Data loading functions defined")

## 5. Metrics Calculation Functions

In [ ]:
def calculate_dice_coefficient(pred: np.ndarray, gt: np.ndarray, class_id: int) -> float:
    """Calculate Dice coefficient for a specific class."""
    pred_mask = (pred == class_id)
    gt_mask = (gt == class_id)
    
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = pred_mask.sum() + gt_mask.sum()
    
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    
    dice = (2.0 * intersection) / union
    return dice


def calculate_iou(pred: np.ndarray, gt: np.ndarray, class_id: int) -> float:
    """Calculate Intersection over Union for a specific class."""
    pred_mask = (pred == class_id)
    gt_mask = (gt == class_id)
    
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    
    iou = intersection / union
    return iou


def calculate_precision_recall(pred: np.ndarray, gt: np.ndarray, class_id: int) -> Tuple[float, float]:
    """Calculate precision and recall for a specific class."""
    pred_mask = (pred == class_id)
    gt_mask = (gt == class_id)
    
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    return precision, recall


def calculate_specificity(pred: np.ndarray, gt: np.ndarray, class_id: int) -> float:
    """Calculate specificity for a specific class."""
    pred_mask = (pred == class_id)
    gt_mask = (gt == class_id)
    
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    return specificity


def calculate_hausdorff_distance(pred: np.ndarray, gt: np.ndarray, class_id: int, percentile: int = 95) -> float:
    """Calculate Hausdorff Distance at given percentile for a specific class."""
    pred_mask = (pred == class_id).astype(np.uint8)
    gt_mask = (gt == class_id).astype(np.uint8)
    
    # Find boundary points
    from scipy.ndimage import binary_erosion
    
    pred_boundary = pred_mask - binary_erosion(pred_mask)
    gt_boundary = gt_mask - binary_erosion(gt_mask)
    
    pred_points = np.argwhere(pred_boundary)
    gt_points = np.argwhere(gt_boundary)
    
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0.0 if len(pred_points) == len(gt_points) == 0 else float('inf')
    
    # Calculate distances
    from scipy.spatial.distance import cdist
    distances_pred_to_gt = cdist(pred_points, gt_points).min(axis=1)
    distances_gt_to_pred = cdist(gt_points, pred_points).min(axis=1)
    
    # Percentile-based Hausdorff
    hd_pred_to_gt = np.percentile(distances_pred_to_gt, percentile)
    hd_gt_to_pred = np.percentile(distances_gt_to_pred, percentile)
    
    hausdorff = max(hd_pred_to_gt, hd_gt_to_pred)
    
    return hausdorff


def calculate_average_surface_distance(pred: np.ndarray, gt: np.ndarray, class_id: int) -> float:
    """Calculate Average Surface Distance for a specific class."""
    pred_mask = (pred == class_id).astype(np.uint8)
    gt_mask = (gt == class_id).astype(np.uint8)
    
    # Find boundary points
    from scipy.ndimage import binary_erosion
    
    pred_boundary = pred_mask - binary_erosion(pred_mask)
    gt_boundary = gt_mask - binary_erosion(gt_mask)
    
    pred_points = np.argwhere(pred_boundary)
    gt_points = np.argwhere(gt_boundary)
    
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0.0 if len(pred_points) == len(gt_points) == 0 else float('inf')
    
    # Calculate distances
    from scipy.spatial.distance import cdist
    distances_pred_to_gt = cdist(pred_points, gt_points).min(axis=1)
    distances_gt_to_pred = cdist(gt_points, pred_points).min(axis=1)
    
    # Average surface distance
    asd = (distances_pred_to_gt.sum() + distances_gt_to_pred.sum()) / (len(pred_points) + len(gt_points))
    
    return asd


def calculate_all_metrics(pred: np.ndarray, gt: np.ndarray, num_classes: int) -> Dict:
    """Calculate all metrics for all classes."""
    metrics = {
        'overall': {},
        'per_class': {}
    }
    
    # Overall pixel accuracy
    metrics['overall']['pixel_accuracy'] = (pred == gt).mean()
    
    # Per-class metrics
    for class_id in range(num_classes):
        class_metrics = {}
        
        class_metrics['dice'] = calculate_dice_coefficient(pred, gt, class_id)
        class_metrics['iou'] = calculate_iou(pred, gt, class_id)
        
        precision, recall = calculate_precision_recall(pred, gt, class_id)
        class_metrics['precision'] = precision
        class_metrics['recall'] = recall
        class_metrics['f1_score'] = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        class_metrics['specificity'] = calculate_specificity(pred, gt, class_id)
        
        # Boundary metrics (skip background)
        if class_id > 0:
            class_metrics['hausdorff_95'] = calculate_hausdorff_distance(pred, gt, class_id, percentile=95)
            class_metrics['avg_surface_distance'] = calculate_average_surface_distance(pred, gt, class_id)
        
        # Pixel counts
        class_metrics['gt_pixels'] = (gt == class_id).sum()
        class_metrics['pred_pixels'] = (pred == class_id).sum()
        
        metrics['per_class'][class_id] = class_metrics
    
    # Overall tissue metrics (all classes except background)
    tissue_dice_scores = [metrics['per_class'][i]['dice'] for i in range(1, num_classes)]
    metrics['overall']['mean_tissue_dice'] = np.mean(tissue_dice_scores)
    
    return metrics

print("✓ Metrics calculation functions defined")

## 6. Load Model

In [ ]:
# Load model
checkpoint = torch.load(Config.MODEL_PATH, map_location='cpu', weights_only=False)
num_classes = checkpoint['model_state_dict']['final_conv.weight'].shape[0]

model = UNetWithPretrainedEncoder(
    encoder_name=Config.ENCODER_NAME,
    num_classes=num_classes
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(Config.DEVICE)
model.eval()

print(f"✓ Model loaded from: {Config.MODEL_PATH}")
print(f"  Number of classes: {num_classes}")
print(f"  Training Val Loss: {checkpoint.get('val_loss', 'N/A')}")
print(f"  Training Val Acc: {checkpoint.get('val_acc', 'N/A')}")

## 7. Run Inference and Calculate Metrics

In [ ]:
@torch.no_grad()
def run_inference(npz_path: Path, model: nn.Module, device: torch.device, input_size: Tuple[int, int]) -> Dict:
    """Run inference on a single sample and calculate all metrics."""
    
    # Load data
    result = extract_m11_and_mask(npz_path)
    if result is None:
        return None
    
    m11_original, gt_mask = result
    original_shape = m11_original.shape
    
    # Preprocess
    input_tensor = preprocess_m11(m11_original, input_size)
    input_tensor = input_tensor.to(device)
    
    # Inference
    logits = model(input_tensor)
    pred_classes = torch.argmax(logits, dim=1)
    
    # Resize back to original size
    pred_classes = F.interpolate(
        pred_classes.unsqueeze(1).float(),
        size=original_shape,
        mode='nearest'
    ).squeeze().long()
    
    pred_mask = pred_classes.cpu().numpy()
    
    # Calculate metrics
    metrics = calculate_all_metrics(pred_mask, gt_mask, num_classes)
    
    return {
        'm11': m11_original,
        'ground_truth': gt_mask,
        'prediction': pred_mask,
        'metrics': metrics
    }

print("✓ Inference function defined")

## 8. Load Data Split and Run Batch Analysis

In [ ]:
# Load data split
with open(Config.DATA_SPLIT_PATH, 'r') as f:
    data_split = json.load(f)

print("=" * 80)
print("DATA SPLIT INFORMATION")
print("=" * 80)
print(f"Train samples: {len(data_split['train'])}")
print(f"Val samples: {len(data_split['val'])}")
print(f"Test samples: {len(data_split['test'])}")
print(f"Isolated samples: {len(data_split.get('isolated', []))}")
print("=" * 80)

# Run inference on isolated samples
print("\nRunning inference on ISOLATED samples...")
isolated_results = []

for sample_info in tqdm(data_split.get('isolated', []), desc="Isolated samples"):
    npz_path = Path(sample_info['path'])
    result = run_inference(npz_path, model, Config.DEVICE, Config.INPUT_SIZE)
    
    if result is not None:
        result['sample_name'] = sample_info['name']
        result['sample_path'] = str(npz_path)
        isolated_results.append(result)
    else:
        print(f"Failed to process: {sample_info['name']}")

print(f"✓ Processed {len(isolated_results)} isolated samples")

# Run inference on test samples
print("\nRunning inference on TEST samples...")
test_results = []

for sample_info in tqdm(data_split['test'], desc="Test samples"):
    npz_path = Path(sample_info['path'])
    result = run_inference(npz_path, model, Config.DEVICE, Config.INPUT_SIZE)
    
    if result is not None:
        result['sample_name'] = sample_info['name']
        result['sample_path'] = str(npz_path)
        test_results.append(result)
    else:
        print(f"Failed to process: {sample_info['name']}")

print(f"✓ Processed {len(test_results)} test samples")
print("\n" + "=" * 80)

## 9. Generate Tables

### Table 1: Overall Segmentation Performance

In [ ]:
def calculate_summary_stats(results: List[Dict], metric_path: List[str]) -> Tuple[float, float]:
    """Calculate mean and std for a metric across all samples."""
    values = []
    for result in results:
        val = result['metrics']
        for key in metric_path:
            val = val[key]
        values.append(val)
    return np.mean(values), np.std(values)


# Calculate overall statistics
table1_data = []

metrics_to_report = [
    ('Overall DSC (Tissue)', ['overall', 'mean_tissue_dice']),
    ('Pixel Accuracy', ['overall', 'pixel_accuracy']),
]

# Add per-class Dice
for class_id, class_name in enumerate(Config.CLASS_NAMES[1:], start=1):  # Skip background
    metrics_to_report.append((f'{class_name} DSC', ['per_class', class_id, 'dice']))

for metric_name, metric_path in metrics_to_report:
    iso_mean, iso_std = calculate_summary_stats(isolated_results, metric_path)
    test_mean, test_std = calculate_summary_stats(test_results, metric_path)
    
    # Simple t-test for p-value
    from scipy.stats import ttest_ind
    iso_vals = [r['metrics'][metric_path[0]][metric_path[1]] if len(metric_path) == 2 
                else r['metrics'][metric_path[0]][metric_path[1]][metric_path[2]] 
                for r in isolated_results]
    test_vals = [r['metrics'][metric_path[0]][metric_path[1]] if len(metric_path) == 2 
                 else r['metrics'][metric_path[0]][metric_path[1]][metric_path[2]] 
                 for r in test_results]
    _, p_value = ttest_ind(iso_vals, test_vals)
    
    table1_data.append({
        'Metric': metric_name,
        'Isolated Samples': f"{iso_mean:.4f} ± {iso_std:.4f}",
        'Test Set': f"{test_mean:.4f} ± {test_std:.4f}",
        'p-value': f"{p_value:.4f}"
    })

table1_df = pd.DataFrame(table1_data)

print("\n" + "=" * 80)
print("TABLE 1: OVERALL SEGMENTATION PERFORMANCE")
print("=" * 80)
print(table1_df.to_string(index=False))
print("=" * 80)

# Save table
table1_df.to_csv(Config.OUTPUT_DIR / 'table1_overall_performance.csv', index=False)
table1_df.to_latex(Config.OUTPUT_DIR / 'table1_overall_performance.tex', index=False)
print(f"\n✓ Table saved to: {Config.OUTPUT_DIR / 'table1_overall_performance.csv'}")

### Table 2: Per-Class Performance (Isolated Samples)

In [ ]:
# Calculate per-class statistics for isolated samples
table2_data = []

for class_id, class_name in enumerate(Config.CLASS_NAMES):
    metrics_names = ['dice', 'iou', 'precision', 'recall', 'f1_score', 'specificity']
    if class_id > 0:  # Add boundary metrics for non-background classes
        metrics_names.extend(['hausdorff_95', 'avg_surface_distance'])
    
    row_data = {'Class': class_name}
    
    for metric in metrics_names:
        values = [r['metrics']['per_class'][class_id][metric] for r in isolated_results]
        mean_val = np.mean(values)
        std_val = np.std(values)
        row_data[metric.replace('_', ' ').title()] = f"{mean_val:.4f} ± {std_val:.4f}"
    
    table2_data.append(row_data)

table2_df = pd.DataFrame(table2_data)

print("\n" + "=" * 80)
print("TABLE 2: PER-CLASS PERFORMANCE (ISOLATED SAMPLES)")
print("=" * 80)
print(table2_df.to_string(index=False))
print("=" * 80)

# Save table
table2_df.to_csv(Config.OUTPUT_DIR / 'table2_per_class_performance.csv', index=False)
table2_df.to_latex(Config.OUTPUT_DIR / 'table2_per_class_performance.tex', index=False)
print(f"\n✓ Table saved to: {Config.OUTPUT_DIR / 'table2_per_class_performance.csv'}")

### Table 3: Per-Sample Detailed Results (Isolated Samples)

In [ ]:
# Per-sample results
table3_data = []

for result in isolated_results:
    row = {
        'Sample Name': result['sample_name'],
        'Overall DSC': f"{result['metrics']['overall']['mean_tissue_dice']:.4f}",
        'Pixel Accuracy': f"{result['metrics']['overall']['pixel_accuracy']:.4f}"
    }
    
    # Add per-class DSC
    for class_id, class_name in enumerate(Config.CLASS_NAMES[1:], start=1):
        row[f'{class_name} DSC'] = f"{result['metrics']['per_class'][class_id]['dice']:.4f}"
    
    # Add boundary metrics for tissue classes
    for class_id, class_name in enumerate(Config.CLASS_NAMES[1:], start=1):
        hd = result['metrics']['per_class'][class_id]['hausdorff_95']
        asd = result['metrics']['per_class'][class_id]['avg_surface_distance']
        row[f'{class_name} HD95'] = f"{hd:.2f}" if hd != float('inf') else "N/A"
        row[f'{class_name} ASD'] = f"{asd:.2f}" if asd != float('inf') else "N/A"
    
    table3_data.append(row)

table3_df = pd.DataFrame(table3_data)

print("\n" + "=" * 80)
print("TABLE 3: PER-SAMPLE DETAILED RESULTS (ISOLATED SAMPLES)")
print("=" * 80)
print(table3_df.to_string(index=False))
print("=" * 80)

# Save table
table3_df.to_csv(Config.OUTPUT_DIR / 'table3_per_sample_results.csv', index=False)
table3_df.to_latex(Config.OUTPUT_DIR / 'table3_per_sample_results.tex', index=False)
print(f"\n✓ Table saved to: {Config.OUTPUT_DIR / 'table3_per_sample_results.csv'}")

### Table 4: Tissue Area Comparison (Ground Truth vs. Predicted)

In [ ]:
# Tissue area comparison
table4_data = []

for result in isolated_results:
    row = {'Sample Name': result['sample_name']}
    
    for class_id, class_name in enumerate(Config.CLASS_NAMES[1:], start=1):  # Skip background
        gt_pixels = result['metrics']['per_class'][class_id]['gt_pixels']
        pred_pixels = result['metrics']['per_class'][class_id]['pred_pixels']
        
        if gt_pixels > 0:
            diff_pct = ((pred_pixels - gt_pixels) / gt_pixels) * 100
        else:
            diff_pct = 0.0 if pred_pixels == 0 else float('inf')
        
        row[f'GT {class_name} (px)'] = int(gt_pixels)
        row[f'Pred {class_name} (px)'] = int(pred_pixels)
        row[f'{class_name} Diff (%)'] = f"{diff_pct:+.2f}" if diff_pct != float('inf') else "N/A"
    
    table4_data.append(row)

table4_df = pd.DataFrame(table4_data)

print("\n" + "=" * 80)
print("TABLE 4: TISSUE AREA COMPARISON (GROUND TRUTH VS. PREDICTED)")
print("=" * 80)
print(table4_df.to_string(index=False))
print("=" * 80)

# Save table
table4_df.to_csv(Config.OUTPUT_DIR / 'table4_tissue_area_comparison.csv', index=False)
table4_df.to_latex(Config.OUTPUT_DIR / 'table4_tissue_area_comparison.tex', index=False)
print(f"\n✓ Table saved to: {Config.OUTPUT_DIR / 'table4_tissue_area_comparison.csv'}")

## 10. Failure Case Analysis

In [ ]:
# Identify best, worst, and median samples based on overall DSC
isolated_results_sorted = sorted(isolated_results, 
                                 key=lambda x: x['metrics']['overall']['mean_tissue_dice'])

worst_sample = isolated_results_sorted[0]
best_sample = isolated_results_sorted[-1]
median_idx = len(isolated_results_sorted) // 2
median_sample = isolated_results_sorted[median_idx]

print("\n" + "=" * 80)
print("FAILURE CASE ANALYSIS")
print("=" * 80)
print(f"\nBest Sample: {best_sample['sample_name']}")
print(f"  Overall DSC: {best_sample['metrics']['overall']['mean_tissue_dice']:.4f}")
print(f"\nMedian Sample: {median_sample['sample_name']}")
print(f"  Overall DSC: {median_sample['metrics']['overall']['mean_tissue_dice']:.4f}")
print(f"\nWorst Sample: {worst_sample['sample_name']}")
print(f"  Overall DSC: {worst_sample['metrics']['overall']['mean_tissue_dice']:.4f}")
print("\n" + "=" * 80)


def visualize_sample_comparison(sample: Dict, title_prefix: str, save_path: Path):
    """Create publication-quality visualization for a sample."""
    m11 = sample['m11']
    gt = sample['ground_truth']
    pred = sample['prediction']
    
    # Create figure with 5 columns
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    # 1. M11 image
    axes[0].imshow(m11, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title('(A) M11 Image', fontweight='bold')
    axes[0].axis('off')
    
    # 2. Ground truth (color-coded)
    gt_rgb = np.zeros((*gt.shape, 3), dtype=np.uint8)
    for class_id, color in Config.CLASS_COLORS.items():
        gt_rgb[gt == class_id] = color
    axes[1].imshow(gt_rgb)
    axes[1].set_title('(B) Ground Truth', fontweight='bold')
    axes[1].axis('off')
    
    # 3. Prediction (color-coded)
    pred_rgb = np.zeros((*pred.shape, 3), dtype=np.uint8)
    for class_id, color in Config.CLASS_COLORS.items():
        pred_rgb[pred == class_id] = color
    axes[2].imshow(pred_rgb)
    axes[2].set_title('(C) Prediction', fontweight='bold')
    axes[2].axis('off')
    
    # 4. Error map
    error_map = np.zeros((*pred.shape, 3), dtype=np.uint8)
    correct = (pred == gt)
    error_map[correct] = [0, 255, 0]  # Green for correct
    error_map[~correct] = [255, 0, 0]  # Red for incorrect
    axes[3].imshow(error_map)
    axes[3].set_title('(D) Error Map\n(Green=Correct, Red=Error)', fontweight='bold')
    axes[3].axis('off')
    
    # 5. Overlay on M11
    axes[4].imshow(m11, cmap='gray', vmin=0, vmax=1)
    overlay = pred_rgb.copy()
    overlay_alpha = np.zeros((*pred.shape, 4))
    tissue_mask = pred > 0
    overlay_alpha[tissue_mask, :3] = overlay[tissue_mask] / 255.0
    overlay_alpha[tissue_mask, 3] = 0.5
    axes[4].imshow(overlay_alpha)
    axes[4].set_title('(E) Overlay', fontweight='bold')
    axes[4].axis('off')
    
    # Add overall title with metrics
    dsc = sample['metrics']['overall']['mean_tissue_dice']
    acc = sample['metrics']['overall']['pixel_accuracy']
    fig.suptitle(f"{title_prefix}: {sample['sample_name']}\nDSC={dsc:.4f}, Accuracy={acc:.4f}",
                 fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved: {save_path}")


# Generate visualizations
print("\nGenerating failure case visualizations...")
visualize_sample_comparison(best_sample, "Best Case", 
                           Config.OUTPUT_DIR / 'failure_analysis_best.png')
visualize_sample_comparison(median_sample, "Median Case", 
                           Config.OUTPUT_DIR / 'failure_analysis_median.png')
visualize_sample_comparison(worst_sample, "Worst Case", 
                           Config.OUTPUT_DIR / 'failure_analysis_worst.png')

print("\n✓ Failure case analysis complete")

## 11. Summary Statistics Export

In [ ]:
# Export comprehensive summary
summary = {
    'dataset': {
        'num_isolated_samples': len(isolated_results),
        'num_test_samples': len(test_results),
        'num_classes': num_classes,
        'class_names': Config.CLASS_NAMES
    },
    'isolated_samples': {
        'sample_names': [r['sample_name'] for r in isolated_results],
        'best_sample': best_sample['sample_name'],
        'worst_sample': worst_sample['sample_name'],
        'median_sample': median_sample['sample_name']
    },
    'files_generated': [
        'table1_overall_performance.csv',
        'table2_per_class_performance.csv',
        'table3_per_sample_results.csv',
        'table4_tissue_area_comparison.csv',
        'failure_analysis_best.png',
        'failure_analysis_median.png',
        'failure_analysis_worst.png'
    ]
}

with open(Config.OUTPUT_DIR / 'analysis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\nAll results saved to: {Config.OUTPUT_DIR}")
print("\nGenerated files:")
for file in summary['files_generated']:
    print(f"  - {file}")
print("\n" + "=" * 80)